# Ablation Study

In this notebook we run an ablation study, where we take each study-trial-seed results and compute the test accuracy of the single antenna gaussian VAEs.

The full model weights from the launch are required for this notebook.

## Setup

In [50]:
import json
from pathlib import Path

import numpy as np
import optuna
import pandas as pd
import torch

from csi_vae.jobs import JobSettings, dataset, fusion, vae
from csi_vae.jobs.evaluator import Evaluator
from csi_vae.jobs.job import make_dataloader

settings = JobSettings()

LAUNCH_DIR = Path("../out/flat_conv_full")
WEIGHTS_DIR = Path("../weights")
ABLATION_STUDY_DIR = LAUNCH_DIR / "ablation_study"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ABLATION_STUDY_DIR.mkdir(exist_ok=True)

In [5]:
train_ds, val_ds, test_ds = dataset.load(
    dataset_path=Path("../") / settings.dataset_path,
    window_size=settings.window_size,
    n_activities=settings.n_activities,
    stride=settings.stride,
)

## Load Studies

We need the studies info to get the model configuration, i.e. the convolutional layer specification that was used.

In [6]:
studies_files = sorted([f.name for f in LAUNCH_DIR.iterdir() if f.is_file() and f.suffix == ".sqlite"])
studies = [
    optuna.load_study(study_name=study.split(".")[0], storage=f"sqlite:///{LAUNCH_DIR / study}").trials_dataframe()
    for study in studies_files
]

## Run Ablation Study

In [ ]:
def load_gaussians(model_report: dict, weights_path: Path) -> list[vae.SingleAntenna]:
    """Load the best model from the given model report and weights path.

    Arguments:
        model_report: A dictionary containing the model information.
        weights_path: Path to the model weights file.

    Returns:
        A list of loaded SingleAntenna models.

    """
    gaussians = [
        vae.SingleAntenna(
            settings.window_size,
            settings.n_subcarriers,
            model_report["n_gaussians"],
            vae.CONV_SPECS[model_report["params"]["conv_layers_spec"]],
        ).to(DEVICE)
        for _ in range(settings.n_antennas)
    ]
    delayed_fusion = fusion.Delayed(
        gaussians,
        model_report["n_gaussians"],
        settings.n_activities,
        model_report["params"]["n_fusion_layers"],
        model_report["params"]["fusion_dropout"],
    ).to(DEVICE)
    delayed_fusion.load_state_dict(torch.load(weights_path))

    return gaussians


def train_and_eval_antenna(
    antenna_idx: int,
    gaussian: vae.SingleAntenna,
    n_gaussians: int,
    model_report: dict,
    seed: int,
) -> float:
    """Train and evaluate a single antenna model.

    Arguments:
        antenna_idx: Index of the antenna to train and evaluate.
        gaussian: The SingleAntenna model to train and evaluate.
        n_gaussians: The number of Gaussian components in the model.
        model_report: A dictionary containing the model information.
        seed: Random seed for reproducibility.

    Returns:
        The accuracy of the model on the test dataset.

    """
    antenna_train_ds = dataset.SingleAntenna(train_ds, antenna_idx)
    antenna_val_ds = dataset.SingleAntenna(val_ds, antenna_idx)
    antenna_test_ds = dataset.SingleAntenna(test_ds, antenna_idx)

    train_dl = make_dataloader(antenna_train_ds, settings.batch_size, shuffle=True, seed=int(seed))
    val_dl = make_dataloader(antenna_val_ds, settings.batch_size, shuffle=False, seed=int(seed))
    test_dl = make_dataloader(antenna_test_ds, settings.batch_size, shuffle=False, seed=int(seed))

    delayed_fusion = fusion.Delayed(
        [gaussian],
        n_gaussians,
        settings.n_activities,
        model_report["params"]["n_fusion_layers"],
        model_report["params"]["fusion_dropout"],
    ).to(DEVICE)

    trainer = fusion.Trainer(
        delayed_fusion,
        train_dl,
        val_dl,
        fusion.TrainerParams(
            model_report["params"]["lr"],
            settings.early_stop_patience,
            settings.early_stop_warmup_epochs,
        ),
        DEVICE,
    )
    trainer.train(settings.n_epochs)

    return Evaluator(delayed_fusion, test_dl, DEVICE).evaluate()

The following code is a mess, and also saves the ablation studies in a weird format

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

for i, study in enumerate(studies):
    completed = study[study["state"] == "COMPLETE"].copy()
    study_results = {}

    for _, trial in completed.iterrows():
        trial_results = {}

        for seed in trial["user_attrs_accuracies"]:
            seed_results = []

            model_report = {
                "n_gaussians": i + 1,
                "params": {
                    "conv_layers_spec": trial["params_conv_layers_spec"],
                    "n_fusion_layers": trial["params_n_fusion_layers"],
                    "fusion_dropout": trial["params_fusion_dropout"],
                    "lr": trial["params_lr"],
                },
            }
            weights_path = WEIGHTS_DIR / f"l{i + 1}" / f"t{trial['number']}" / f"s{seed}" / "delayed_fusion.pt"
            gaussians = load_gaussians(model_report, weights_path)

            seed_results = [0.0] * settings.n_antennas
            with ThreadPoolExecutor(max_workers=settings.n_antennas) as executor:
                futures = {
                    executor.submit(
                        train_and_eval_antenna,
                        antenna_idx,
                        gaussians[antenna_idx],
                        i + 1,
                        model_report,
                        seed,
                    ): antenna_idx
                    for antenna_idx in range(settings.n_antennas)
                }

                for future in as_completed(futures):
                    antenna_idx = futures[future]
                    seed_results[antenna_idx] = future.result()

            trial_results[str(seed)] = seed_results
            print(f"Completed seed {seed} of trial {trial['number']} with accuracies: {seed_results}")

        study_results[str(trial["number"])] = trial_results

    study_results = pd.DataFrame(study_results)
    study_results.to_csv(ABLATION_STUDY_DIR / f"l{i + 1}.csv", index=False)

## Plot Results

In [ ]:
ablation_results_files = sorted([result.name for result in ABLATION_STUDY_DIR.iterdir() if result.is_file()])

ablation_results = []
for result_file in ablation_results_files:
    full_df = pd.read_csv(ABLATION_STUDY_DIR / result_file)

    study_results = []
    for col in range(full_df.shape[1]):
        seed_results = full_df.iloc[:, col].dropna().apply(json.loads)
        seed_results = np.array(seed_results.tolist())  # (n_seeds, n_antennas)

        seed_medians = np.median(seed_results, axis=0)  # (n_antennas,)
        study_results.append(seed_medians)

    study_results = np.array(study_results)  # (n_trials, n_antennas)
    print(study_results.shape)
    ablation_results.append(study_results)

ablation_results = np.array(ablation_results)  # (n_gaussians, n_trials, n_antennas)
print(ablation_results.shape)

(100, 4)
(1, 100, 4)


In [ ]:
studies_files = sorted([f.name for f in LAUNCH_DIR.iterdir() if f.is_file() and f.suffix == ".sqlite"])
studies = [
    optuna.load_study(study_name=study.split(".")[0], storage=f"sqlite:///{LAUNCH_DIR / study}").trials_dataframe()
    for study in studies_files
]

In [ ]:
for i, study in enumerate(studies):
    completed = study[study["state"] == "COMPLETE"].copy()

    
    best_models_per_study: list[_ModelReport] = []

    for i, study in enumerate(studies):
        completed = study[study["state"] == "COMPLETE"].copy()
        study_best: _ModelReport = _ModelReport(
            n_gaussians=i + 1,
            trial_number=0,
            trial_value=0.0,
            seed=0,
            best_seed_accuracy=0.0,
            params={},
        )
        study_best_trial = 0
        study_best_trial_value = 0.0

        for _, trial in completed.iterrows():
            accuracies_per_seed = trial["user_attrs_accuracies"]

            best_seed = max(accuracies_per_seed, key=accuracies_per_seed.get)
            best_accuracy = float(accuracies_per_seed[str(best_seed)])

            if trial["value"] > study_best["trial_value"]:
                study_best = _ModelReport(
                    n_gaussians=i + 1,
                    trial_number=trial["number"],
                    trial_value=trial["value"],
                    seed=int(best_seed),
                    best_seed_accuracy=best_accuracy,
                    params=trial.filter(like="params_").rename(lambda x: x.replace("params_", "")).to_dict(),
                )

        best_models_per_study.append(study_best)